In [1]:
import pandas as pd
import tldextract
from urllib.parse import urlparse

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

http_path = "/home/arch/Documents/Paper 1/CERT-Insider-Threat-Dataset/r4.2/http.csv"

tranco_path = "/home/arch/Documents/Paper 1/CERT-Insider-Threat-Dataset/r4.2/blacklist_urls/tranco_top1m.csv"

phishtank_path = "/home/arch/Documents/Paper 1/CERT-Insider-Threat-Dataset/r4.2/blacklist_urls/verified_online.csv"

# ------------------------------------------------------------
# Read only first 100,000 CERT records
# ------------------------------------------------------------

http_chunk = next(
    pd.read_csv(
        http_path,
        chunksize=100_000,
        low_memory=False
    )
)

print(http_chunk.shape)

# ------------------------------------------------------------
# Read Tranco
# ------------------------------------------------------------

tranco = pd.read_csv(
    tranco_path,
    header=None,
    names=["rank", "domain"]
)

tranco_domains = set(
    tranco["domain"]
    .str.lower()
    .str.strip()
)

print("Tranco domains:", len(tranco_domains))

# ------------------------------------------------------------
# Read PhishTank
# ------------------------------------------------------------

phish = pd.read_csv(
    phishtank_path,
    low_memory=False
)

print("PhishTank URLs:", len(phish))

# ------------------------------------------------------------
# Domain extraction (for Tranco only)
# ------------------------------------------------------------

extractor = tldextract.TLDExtract(suffix_list_urls=None)

def registered_domain(url):
    try:
        ext = extractor(url)

        if ext.domain == "":
            return None

        return f"{ext.domain}.{ext.suffix}".lower()

    except Exception:
        return None

# ------------------------------------------------------------
# URL normalization (for PhishTank)
# ------------------------------------------------------------

def normalize_url(url):

    try:

        parsed = urlparse(str(url))

        host = parsed.hostname

        if host is None:
            return None

        host = host.lower()

        if host.startswith("www."):
            host = host[4:]

        path = parsed.path.rstrip("/")

        return host + path

    except Exception:
        return None

# ------------------------------------------------------------
# Extract CERT features
# ------------------------------------------------------------

http_chunk["registered_domain"] = http_chunk["url"].apply(registered_domain)

http_chunk["normalized_url"] = http_chunk["url"].apply(normalize_url)

# ------------------------------------------------------------
# Prepare PhishTank lookup
# ------------------------------------------------------------

phish["normalized_url"] = phish["url"].apply(normalize_url)

phishtank_urls = set(
    phish["normalized_url"]
    .dropna()
    .unique()
)

print("Unique normalized phishing URLs:", len(phishtank_urls))

# ------------------------------------------------------------
# Feature engineering
# ------------------------------------------------------------

http_chunk["tranco_flag"] = (
    http_chunk["registered_domain"]
    .isin(tranco_domains)
    .astype("uint8")
)

http_chunk["phishtank_flag"] = (
    http_chunk["normalized_url"]
    .isin(phishtank_urls)
    .astype("uint8")
)

# ------------------------------------------------------------
# Combined reputation
# ------------------------------------------------------------

def url_reputation(row):

    if row["phishtank_flag"] == 1:
        return "Malicious"

    if row["tranco_flag"] == 1:
        return "Benign"

    return "Unknown"

http_chunk["url_reputation"] = http_chunk.apply(url_reputation, axis=1)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

display(
    http_chunk[
        [
            "url",
            "registered_domain",
            "normalized_url",
            "tranco_flag",
            "phishtank_flag",
            "url_reputation"
        ]
    ].head(30)
)

print()

print("Benign :", (http_chunk["url_reputation"] == "Benign").sum())
print("Malicious :", (http_chunk["url_reputation"] == "Malicious").sum())
print("Unknown :", (http_chunk["url_reputation"] == "Unknown").sum())

(100000, 6)
Tranco domains: 1000000
PhishTank URLs: 69426
Unique normalized phishing URLs: 63622


,url,registered_domain,normalized_url,tranco_flag,phishtank_flag,url_reputation
0,http://msn.com/The_Human_Centipede_First_Seque...,msn.com,msn.com/The_Human_Centipede_First_Sequence/kat...,1,0,Benign
1,http://urbanspoon.com/Plunketts_Creek_Loyalsoc...,urbanspoon.com,urbanspoon.com/Plunketts_Creek_Loyalsock_Creek...,1,0,Benign
2,http://aa.com/Rhodocene/rhodocenium/fhaavatqrf...,aa.com,aa.com/Rhodocene/rhodocenium/fhaavatqrfxgbcrkr...,1,0,Benign
3,http://groupon.com/Leonhard_Euler/leonhard/tne...,groupon.com,groupon.com/Leonhard_Euler/leonhard/tneqravafr...,1,0,Benign
4,http://flickr.com/Inauguration_of_Barack_Obama...,flickr.com,flickr.com/Inauguration_of_Barack_Obama/biden/...,1,0,Benign
5,http://skype.com/William_D_Boyce/lsa/onpxcnpxp...,skype.com,skype.com/William_D_Boyce/lsa/onpxcnpxpurzvfge...,1,0,Benign
6,http://wikipedia.org/Maya_MIA_album/rusko/obju...,wikipedia.org,wikipedia.org/Maya_MIA_album/rusko/objuhagvatp...,1,0,Benign
7,http://constantcontact.com/2008_ACC_Championsh...,constantcontact.com,constantcontact.com/2008_ACC_Championship_Game...,1,0,Benign
8,http://tribalfusion.com/Abbey_Theatre/classon/...,tribalfusion.com,tribalfusion.com/Abbey_Theatre/classon/pheevph...,1,0,Benign
9,http://pcworld.com/Alexandre_Banza/bokassa/bcg...,pcworld.com,pcworld.com/Alexandre_Banza/bokassa/bcgvzvmngv...,1,0,Benign



Benign : 97859
Malicious : 0
Unknown : 2141


In [2]:
unknown_urls = http_chunk[http_chunk["url_reputation"] == "Unknown"]

In [3]:
unknown_urls

,id,date,user,pc,url,content,registered_domain,normalized_url,tranco_flag,phishtank_flag,url_reputation
99,{F7R1-H3PU31BA-8717ZLNZ},01/02/2010 07:54:27,RZC0746,PC-7500,http://sparkstudios.com/French_colonization_of...,replace reduced age havilland were damaged lea...,sparkstudios.com,sparkstudios.com/French_colonization_of_Texas/...,0,0,Unknown
101,{R6V9-N1KK77SD-8938BXSN},01/02/2010 07:54:37,LRR0148,PC-4275,http://1saleaday.com/Mercury_dime/dimes/frphev...,weeks i hospital myers waverly fact potential ...,1saleaday.com,1saleaday.com/Mercury_dime/dimes/frphevglsvern...,0,0,Unknown
241,{L4R3-S6YF69UD-6707MWEE},01/02/2010 08:18:16,AJR0319,PC-4736,http://1saleaday.com/Mercury_dime/dimes/frphev...,off described regina number murder dangerous p...,1saleaday.com,1saleaday.com/Mercury_dime/dimes/frphevglsvern...,0,0,Unknown
562,{M3A0-K7QG99ZW-9512GIDL},01/02/2010 08:59:05,ATE0869,PC-1313,http://1saleaday.com/Mercury_dime/dimes/frphev...,section kennedy mississippi emptied milepost f...,1saleaday.com,1saleaday.com/Mercury_dime/dimes/frphevglsvern...,0,0,Unknown
656,{A7S7-H3JM42OH-5399UFSE},01/02/2010 09:13:17,ATE0869,PC-1313,http://sparkstudios.com/French_colonization_of...,factory bad scored substantive 47 myself yugos...,sparkstudios.com,sparkstudios.com/French_colonization_of_Texas/...,0,0,Unknown
...,...,...,...,...,...,...,...,...,...,...,...
99802,{S0E0-J0ZS98QW-6033XHVO},01/05/2010 08:22:57,TBT0461,PC-4679,http://ilivid.com/Nostradamus/nostredame/fnsrg...,physical fund account was check out graphics p...,ilivid.com,ilivid.com/Nostradamus/nostredame/fnsrglubgry7...,0,0,Unknown
99888,{P7W5-K8IY77SR-2660HPEE},01/05/2010 08:23:28,BAL0044,PC-5179,http://megaclick.com/Albert_Prince_Consort/cur...,went nine 14 got did 1 32 five coast and seeme...,megaclick.com,megaclick.com/Albert_Prince_Consort/curragh/cy...,0,0,Unknown
99889,{H0P7-N3SN43SH-6858OATK},01/05/2010 08:23:29,AJR0319,PC-4736,http://1saleaday.com/Mercury_dime/dimes/frphev...,murder sheriffs wearing school joint h summer ...,1saleaday.com,1saleaday.com/Mercury_dime/dimes/frphevglsvern...,0,0,Unknown
99917,{W4G0-J9GD06WD-9852OJHS},01/05/2010 08:23:41,AJF0370,PC-8651,http://sparkstudios.com/Hurricane_Kiko_1989/lo...,discarded again concluding the usual next thei...,sparkstudios.com,sparkstudios.com/Hurricane_Kiko_1989/lorena/on...,0,0,Unknown


In [6]:
import pandas as pd
from functools import reduce

# ------------------------------------------------------------
# Prepare datetime columns
# ------------------------------------------------------------

http_chunk["datetime"] = pd.to_datetime(http_chunk["date"])
http_chunk["date_only"] = http_chunk["datetime"].dt.date
http_chunk["hour"] = http_chunk["datetime"].dt.hour

# ------------------------------------------------------------
# URLs considered malicious
# ------------------------------------------------------------

malicious_labels = {"Malicious", "Unknown", "Conflict"}

http_chunk["malicious_flag"] = (
    http_chunk["url_reputation"]
    .isin(malicious_labels)
    .astype("uint8")
)

# ------------------------------------------------------------
# Average URLs visited per day
# ------------------------------------------------------------

daily_urls = (
    http_chunk
    .groupby(["user", "date_only"])
    .size()
    .reset_index(name="urls_per_day")
)

avg_urls_per_day = (
    daily_urls
    .groupby("user")["urls_per_day"]
    .mean()
    .reset_index(name="avg_urls_per_day")
)

# ------------------------------------------------------------
# Malicious URL features
# ------------------------------------------------------------

potentially_malicious_df = http_chunk[
    http_chunk["url_reputation"].isin(["Malicious", "Unknown", "Conflict"])
]

total_potentially_malicious_urls = (
    potentially_malicious_df
    .groupby("user")
    .size()
    .reset_index(name="total_potentially_malicious_urls")
)

potentially_malicious_per_day = (
    potentially_malicious_df
    .groupby(["user", "date_only"])
    .size()
    .reset_index(name="potentially_malicious_urls_per_day")
)

avg_potentially_malicious_urls_per_day = (
    potentially_malicious_per_day
    .groupby("user")["potentially_malicious_urls_per_day"]
    .mean()
    .reset_index(name="avg_potentially_malicious_urls_per_day")
)

# ------------------------------------------------------------
# Average distinct registered domains per day
# ------------------------------------------------------------

daily_domains = (
    http_chunk
    .groupby(["user", "date_only"])["registered_domain"]
    .nunique()
    .reset_index(name="distinct_domains_per_day")
)

avg_distinct_domains_per_day = (
    daily_domains
    .groupby("user")["distinct_domains_per_day"]
    .mean()
    .reset_index(name="avg_distinct_domains_per_day")
)

# ------------------------------------------------------------
# Average after-hours browsing per day
# Before 06:00 or after 18:00
# ------------------------------------------------------------

http_chunk["after_hours"] = (
    (http_chunk["hour"] < 6) |
    (http_chunk["hour"] > 18)
)

after_hours_daily = (
    http_chunk[http_chunk["after_hours"]]
    .groupby(["user", "date_only"])
    .size()
    .reset_index(name="after_hours_count")
)

avg_after_hours_browsing_per_day = (
    after_hours_daily
    .groupby("user")["after_hours_count"]
    .mean()
    .reset_index(name="avg_after_hours_browsing_per_day")
)

# ------------------------------------------------------------
# Merge all features
# ------------------------------------------------------------

feature_dfs = [
    avg_urls_per_day,
    avg_potentially_malicious_urls_per_day,
    total_potentially_malicious_urls,
    avg_distinct_domains_per_day,
    avg_after_hours_browsing_per_day
]

final_features_http = reduce(
    lambda left, right: pd.merge(left, right, on="user", how="outer"),
    feature_dfs
)

final_features_http.fillna(0, inplace=True)

# ------------------------------------------------------------
# Final Risk Score
# ------------------------------------------------------------

final_features_http["final_risk_score"] = (
    (final_features_http["avg_after_hours_browsing_per_day"] > 1).astype(int) * 1 +
    (final_features_http["avg_potentially_malicious_urls_per_day"] > 3).astype(int) * 2 +
    (final_features_http["total_potentially_malicious_urls"] > 50).astype(int) * 2 +
    (final_features_http["total_potentially_malicious_urls"] > 100).astype(int) * 1
)

# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

display(final_features_http.head())

,user,avg_urls_per_day,avg_potentially_malicious_urls_per_day,total_potentially_malicious_urls,avg_distinct_domains_per_day,avg_after_hours_browsing_per_day,final_risk_score
0,AAE0190,143.0,4.0,4.0,52.0,0.0,2
1,AAF0535,29.0,0.0,0.0,15.0,0.0,0
2,AAF0791,48.5,1.0,1.0,22.0,0.0,0
3,AAL0706,10.0,0.0,0.0,8.0,0.0,0
4,AAM0658,29.0,0.0,0.0,19.0,6.0,1


In [7]:
display(final_features_http.head(20))

,user,avg_urls_per_day,avg_potentially_malicious_urls_per_day,total_potentially_malicious_urls,avg_distinct_domains_per_day,avg_after_hours_browsing_per_day,final_risk_score
0,AAE0190,143.000000,4.000000,4.0,52.000000,0.0,2
1,AAF0535,29.000000,0.000000,0.0,15.000000,0.0,0
2,AAF0791,48.500000,1.000000,1.0,22.000000,0.0,0
3,AAL0706,10.000000,0.000000,0.0,8.000000,0.0,0
4,AAM0658,29.000000,0.000000,0.0,19.000000,6.0,1
5,AAN0823,10.000000,4.000000,4.0,6.000000,2.0,3
6,AAS0442,15.500000,0.000000,0.0,8.500000,0.0,0
7,AAV0450,10.000000,0.000000,0.0,5.000000,0.0,0
8,AAW0353,15.500000,0.000000,0.0,8.500000,0.0,0
9,ABC0174,114.333333,6.666667,20.0,37.666667,0.0,2
